In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA04 - Travel, Lodging, and Entertainment (Non-POs > $250)
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Coupa Data: 7 filtered files (ritm16070584_...). Finance data is EXCLUDED.
   
   BUSINESS LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. DATA SOURCE: Uses COUPA FILES ONLY. 
   3. NON-PURCHASE ORDER FILTER: Filters strictly for rows where the Document Type 
      (Col A in Excel) is 'Non PO' (or 'Non-PO').
   4. THRESHOLD RULE: Strictly filters for transactions where the 'Total' (Col D) 
      is greater than 250.
   5. ACCOUNT PARSING: Parses the 'Account' string (format: xxxx-xxxx-xxxxxx) to 
      extract the Cost Center and Category Code. 
   6. TARGET CATEGORY CODES: Filters strictly for the ~30 approved target codes.
   7. THE "793" EXCEPTION RULE: If a Category Code is 793, it is strictly locked to 
      Assessable Unit '101016'. If it maps to any other AU, it is dropped.
   8. FINAL OUTPUT: Rolls up and counts the valid instances, outputting a numerical value.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    -- 2. Pull the mapped Cost Centers from our finalized bootstrap view
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 3. Union all 7 Coupa files into one master dataset. 
    -- Pulls DocumentType for the Non-PO filter, and Total for the >250 filter.
    SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Non_PO_Transactions AS (
    -- 4. Parse the Account string, filter for 'Non PO', and filter Total > 250
    SELECT 
        Account AS Raw_String,
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        TRIM(DocumentType) AS Document_Type,
        -- Safely cast Total to a decimal, removing any potential commas
        TRY_CAST(REPLACE(Total, ',', '') AS DECIMAL(15,2)) AS Transaction_Total
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
      -- EBA04 RULE 1: Document Type must be Non-PO (Accommodates both 'NON PO' and 'NON-PO')
      AND UPPER(TRIM(DocumentType)) IN ('NON PO', 'NON-PO')
      -- EBA04 RULE 2: Value must be strictly greater than 250
      AND TRY_CAST(REPLACE(Total, ',', '') AS DECIMAL(15,2)) > 250
),

Flagged_Engagements AS (
    -- 5. Join mapping, apply the ~30 Category Codes, and enforce the 793 rule
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        m.AU_ID
    FROM Coupa_Non_PO_Transactions t
    INNER JOIN CC_Mapping m ON t.Cleaned_CC = m.Mapped_CC
    WHERE 
        -- RULE 1: Must be in the approved master category list
        t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
        
        -- RULE 2: If the code is 793, it MUST be mapped to AU 101016 to be counted
        AND NOT (t.CatCode_Int = 793 AND m.AU_ID != '101016')
),

Engagements_By_AU AS (
    -- 6. Count the total valid flagged transactions per AU
    SELECT 
        AU_ID,
        COUNT(Raw_String) AS Match_Count
    FROM Flagged_Engagements
    GROUP BY AU_ID
)

-- 7. Final Output: Strictly structured numerical count anchored to Master list
SELECT 
    mast.BusinessID,                           
    mast.AU_Name,                              
    mast.Business_Segment,
    'EBA04' AS QuestionID,               
    -- Numerical Output: Counts the instances. If none, outputs '0'
    COALESCE(CAST(e.Match_Count AS STRING), '0') AS Response,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Engagements_By_AU e 
    ON mast.BusinessID = e.AU_ID
ORDER BY mast.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA04 - Travel, Lodging, and Ent. (Non-POs > $250) Traceability
   
   PURPOSE: 
   Isolates the data flow for EBA04 to show exactly how Coupa records are filtered 
   for 'Non PO', how the > $250 threshold is enforced, whether their Category Codes 
   trigger a flag, how they map to an AU, and whether they get blocked by the "793" rule.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data for bridge verification
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Combined_Coupa_Raw AS (
    -- 2. Stack all 7 Coupa target files together
    SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, DocumentType, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 3. Extract the CC, Category Code, Document Type, and numerical Total
    SELECT 
        Account AS Raw_String,
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        TRIM(DocumentType) AS Document_Type,
        TRY_CAST(REPLACE(Total, ',', '') AS DECIMAL(15,2)) AS Transaction_Total
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

CC_Mapping AS (
    -- 4. Pull the mapped Cost Centers from our finalized bootstrap view
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

-- 5. Output the explicit tracing details for every transaction
SELECT 
    t.Raw_String AS Original_Record,
    t.Cleaned_CC AS Parsed_Cost_Center,
    t.CatCode_Int AS Parsed_Category_Code,
    t.Document_Type,
    t.Transaction_Total,
    
    -- Flag if the Category Code exists in our 30-item master list
    CASE WHEN t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892) 
         THEN 'YES' ELSE 'NO' END AS Is_Target_Category,
         
    m.AU_ID AS Mapped_AU_ID,
    mast.Master_AU_Name,
    
    -- Trace exactly how the rules are applied step-by-step
    CASE 
        WHEN UPPER(TRIM(t.Document_Type)) NOT IN ('NON PO', 'NON-PO') THEN '❌ DROPPED: Not a Non-PO document'
        WHEN t.Transaction_Total <= 250 OR t.Transaction_Total IS NULL THEN '❌ DROPPED: Total is not > 250'
        WHEN t.CatCode_Int NOT IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892) THEN '❌ DROPPED: Not a target category'
        WHEN t.CatCode_Int = 793 AND m.AU_ID != '101016' THEN '❌ BLOCKED: 793 only valid for 101016'
        WHEN t.CatCode_Int = 793 AND m.AU_ID = '101016' THEN '✅ KEPT: Valid 793 mapping'
        ELSE '✅ KEPT: Standard Category mapped'
    END AS Pipeline_Status,
    
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'FAILED BRIDGE' ELSE 'SUCCESS' END AS Master_List_Status
    
FROM Coupa_Parsed t
LEFT JOIN CC_Mapping m 
    ON t.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast 
    ON m.AU_ID = mast.Master_Numeric_ID

-- Optional: Uncomment below to view only the successful records
-- WHERE UPPER(TRIM(t.Document_Type)) IN ('NON PO', 'NON-PO')
--   AND t.Transaction_Total > 250
--   AND t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
ORDER BY t.Cleaned_CC;